# Multilingual Enterprise Search with Compass

This notebook demonstrates cross-lingual enterprise search: ask questions in English against French-language policy documents, retrieve relevant passages with Compass, and generate grounded English answers with citations.

This workflow uses:

- **Compass**: Document parsing, indexing, and chunk retrieval
- **Embed**: Multilingual embeddings for cross-lingual retrieval
- **Rerank**: Precision reranking with a multilingual rerank model
- **Command**: Grounded answer generation in English with citations

## Summary of Steps

1. **Setup** — Install dependencies and load configuration from environment variables.
2. **Prepare or parse documents** — Load or parse a sample corpus of French financial-policy documents.
3. **Create or connect to a Compass index** — Index the documents for multilingual retrieval.
4. **Run English queries against French documents** — Search the index with English queries and inspect returned French chunks.
5. **Rerank for precision** — Rerank retrieved chunks with a multilingual rerank model.
6. **Generate grounded English answers** — Produce a cited English answer with Command.
7. **Next steps** — Adapt the pattern to other languages and enterprise deployments.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "cohere", "cohere-compass-sdk", "python-dotenv", "-q"])

# Setup

Install the required packages and load credentials from environment variables. Do not hardcode API keys, Compass URLs, or bearer tokens in this notebook.

Create a local `.env` file (not committed to git) with:

- `COHERE_API_KEY`
- `COMPASS_INDEX_URL`
- `COMPASS_PARSER_URL` (optional — only needed if parsing real files instead of the synthetic corpus)
- `COMPASS_BEARER_TOKEN` (optional, depending on your Compass deployment)

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=True)

COHERE_API_KEY = os.environ.get("COHERE_API_KEY")
COMPASS_INDEX_URL = os.environ.get("COMPASS_INDEX_URL")
COMPASS_PARSER_URL = os.environ.get("COMPASS_PARSER_URL")
COMPASS_BEARER_TOKEN = os.environ.get("COMPASS_BEARER_TOKEN", "")

required = ["COHERE_API_KEY", "COMPASS_INDEX_URL"]
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Set them in a local .env file before running this notebook."
    )

In [ ]:
import cohere
from cohere_compass.clients import CompassParserClient, CompassClient

co = cohere.ClientV2(api_key=COHERE_API_KEY)

compass_client = CompassClient(
    index_url=COMPASS_INDEX_URL,
    bearer_token=COMPASS_BEARER_TOKEN or None,
)

parser_client = None
if COMPASS_PARSER_URL:
    parser_client = CompassParserClient(
        parser_url=COMPASS_PARSER_URL,
        bearer_token=COMPASS_BEARER_TOKEN or None,
    )

# To parse real files instead of synthetic text, use the parser client:
# parsed_docs = list(parser_client.process_file(
#     index_name=INDEX_NAME,
#     file_path="path/to/your/document.pdf",
# ))
# compass_client.insert_docs(index_name=INDEX_NAME, docs=parsed_docs)

print("✅ All clients initialized successfully")

# Prepare Documents

The next cell defines a publishable sample corpus of seven synthetic French financial-policy documents from the fictional Banque Centrale de Mercure. Each entry includes an `id`, `title`, `source`, and `content` field, covering topics such as inflation, monetary policy, interest rates, regulatory compliance, and financial supervision.

In [ ]:
sample_documents = [
    {
        "id": "doc-001",
        "title": "Position de la Banque Centrale de Mercure sur l'inflation",
        "source": "Politique financière de Mercure, Section 1",
        "content": (
            "La Banque Centrale de Mercure réaffirme que la maîtrise de l'inflation demeure "
            "sa priorité absolue pour préserver la stabilité des prix et la confiance des ménages. "
            "Face à une inflation encore supérieure à l'objectif de 2 %, la direction générale "
            "considère que les pressions inflationnistes proviennent principalement des coûts "
            "énergétiques et des tensions sur les chaînes d'approvisionnement. "
            "La banque centrale indique qu'elle maintiendra une politique monétaire restrictive "
            "tant que les anticipations d'inflation ne convergeront pas durablement vers la cible."
        ),
    },
    {
        "id": "doc-002",
        "title": "Cadre de mesure et de suivi de l'inflation nationale",
        "source": "Politique financière de Mercure, Section 2",
        "content": (
            "Ce document décrit la méthodologie utilisée pour calculer l'indice des prix à la "
            "consommation harmonisé dans l'Union économique de Mercure. "
            "L'indice intègre les variations de prix des biens essentiels, des services et des "
            "loyers, avec une pondération révisée chaque année par l'institut statistique national. "
            "Les autorités publient un rapport trimestriel comparant l'inflation courante à "
            "l'objectif de stabilité des prix fixé par le conseil monétaire. "
            "Tout écart persistant supérieur à un point de pourcentage déclenche une revue "
            "obligatoire des instruments de politique économique."
        ),
    },
    {
        "id": "doc-003",
        "title": "Directive sur les taux d'intérêt et les opérations de refinancement",
        "source": "Politique financière de Mercure, Section 3",
        "content": (
            "Le comité de politique monétaire fixe le taux directeur principal à un niveau "
            "compatible avec le retour graduel de l'inflation vers 2 % à moyen terme. "
            "Les opérations de refinancement hebdomadaires sont menées au taux directeur, "
            "tandis que la facilité de dépôt offre un taux inférieur de 50 points de base. "
            "Les établissements de crédit sont invités à transmettre ces conditions aux "
            "emprunteurs des secteurs réel et immobilier dans un délai maximal de 30 jours. "
            "Toute modification des taux est communiquée simultanément au marché et au public."
        ),
    },
    {
        "id": "doc-004",
        "title": "Obligations de conformité réglementaire pour les établissements financiers",
        "source": "Politique financière de Mercure, Section 4",
        "content": (
            "L'Autorité de supervision du secteur financier de Mercure impose un dispositif "
            "de conformité couvrant la lutte contre le blanchiment de capitaux et le "
            "financement du terrorisme. "
            "Chaque établissement doit nommer un responsable conformité indépendant et "
            "soumettre un rapport annuel attestant le respect des procédures de vérification "
            "d'identité et de surveillance des transactions. "
            "Les manquements graves peuvent entraîner des sanctions administratives, "
            "y compris la restriction d'activités sur certains marchés. "
            "Les institutions ont l'obligation de former l'ensemble du personnel exposé "
            "aux risques de non-conformité au moins une fois par an."
        ),
    },
    {
        "id": "doc-005",
        "title": "Exigences en fonds propres et ratios de solvabilité",
        "source": "Politique financière de Mercure, Section 5",
        "content": (
            "Les banques commerciales agréées doivent maintenir un ratio de fonds propres "
            "CET1 minimal de 10,5 % de leurs engagements pondérés en fonction des risques. "
            "Un coussin de conservation supplémentaire de 2,5 % s'applique aux institutions "
            "systémiquement importantes identifiées chaque année par le régulateur. "
            "Les fonds propres admissibles comprennent les actions ordinaires, les bénéfices "
            "non distribués et certaines obligations subordonnées éligibles. "
            "Un non-respect des seuils impose un plan de redressement approuvé sous 90 jours."
        ),
    },
    {
        "id": "doc-006",
        "title": "Politique de gestion des risques opérationnels et de crédit",
        "source": "Politique financière de Mercure, Section 6",
        "content": (
            "Ce cadre interne définit les processus d'identification, d'évaluation et de "
            "atténuation des risques opérationnels et de crédit au sein des établissements "
            "sous surveillance. "
            "Les directions des risques doivent présenter trimestriellement au conseil "
            "d'administration un tableau de bord consolidé incluant les pertes réalisées, "
            "les expositions aux contreparties et les incidents majeurs. "
            "Les prêts aux secteurs fortement cycliques requièrent une analyse de "
            "solvabilité renforcée et des provisions prudentielles adaptées. "
            "La politique exige également des tests de résistance semestriels sur les "
            "portefeuilles les plus exposés."
        ),
    },
    {
        "id": "doc-007",
        "title": "Gestion de la liquidité et du risque de taux à long terme",
        "source": "Politique financière de Mercure, Section 7",
        "content": (
            "Les établissements doivent maintenir un ratio de liquidité à court terme "
            "supérieur à 100 % et un coefficient de liquidité stable net au-dessus du "
            "minimum réglementaire de 110 %. "
            "La gestion actif-passif doit anticiper l'impact des variations de taux "
            "d'intérêt sur la marge d'intermédiation sur un horizon de cinq ans. "
            "Les trésoreries sont tenues de conserver des actifs liquides de haute qualité "
            "couvrant au moins 30 jours de sorties de fonds stressées. "
            "Tout déficit de liquidité signalé au régulateur déclenche un plan d'urgence "
            "validé par la direction financière."
        ),
    },
]

In [ ]:
print(f"Loaded {len(sample_documents)} documents")
for doc in sample_documents:
    print(f"  - [{doc['id']}] {doc['title']}")

# Create or Connect to a Compass Index

`CompassClient` and `CompassParserClient` were initialized in the Setup section using your environment variables. This section creates a Compass index (if needed), builds `CompassDocument` objects from the sample corpus, and inserts them for multilingual retrieval. Set `INDEX_NAME` in the next cell to match your deployment.

In [ ]:
INDEX_NAME = "your-index-name"  # Replace with your Compass index name

# Create index if it doesn't exist (safe to run multiple times)
try:
    compass_client.create_index(index_name=INDEX_NAME)
    print(f"✅ Index '{INDEX_NAME}' created or already exists")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"ℹ️ Index '{INDEX_NAME}' already exists — continuing")
    else:
        raise

## Building Compass Documents

When inserting documents directly (without the parser), you must manually construct `CompassDocumentChunk` objects. Compass will reject any document with an empty `chunks` list with a 500 error: `"Document has no chunks to index."`

Each chunk requires:
- `chunk_id` — unique per chunk (e.g. `"{doc_id}-chunk-0"`)
- `sort_id` — controls chunk ordering within a document
- `document_id` and `parent_document_id` — both set to the parent doc ID
- `content` — dict of the fields you want indexed
- Set `index_fields` on the parent `CompassDocument` to tell Compass which content fields to embed (e.g. `["text", "title"]`)

> ⚠️ A `CompassDocument` built with only `metadata` + `content` will pass local validation but will be rejected by the Compass server with HTTP 500: `"Document has no chunks to index."`

In [ ]:
from cohere_compass.models.documents import (
    CompassDocument,
    CompassDocumentMetadata,
    CompassDocumentChunk,
)

def build_compass_doc(doc: dict) -> CompassDocument:
    chunk = CompassDocumentChunk(
        chunk_id=f"{doc['id']}-chunk-0",
        sort_id="0000",
        document_id=doc["id"],
        parent_document_id=doc["id"],
        content={
            "title": doc["title"],
            "text": doc["content"],
            "source": doc["source"],
        },
    )
    return CompassDocument(
        metadata=CompassDocumentMetadata(
            document_id=doc["id"],
            filename=doc["title"],
        ),
        content={
            "title": doc["title"],
            "text": doc["content"],
            "source": doc["source"],
        },
        chunks=[chunk],
        index_fields=["text", "title"],
    )

compass_docs = [build_compass_doc(doc) for doc in sample_documents]

## Inserting Documents

`insert_docs()` raises `CompassInsertionError` on failure and returns `None` on success — do NOT wrap it in `list()` or you will get a `TypeError` on successful runs. To debug insertion failures, inspect the error details directly:

```python
except CompassInsertionError as e:
    for err in e.errors:
        print(err)
```

In [ ]:
compass_client.insert_docs(
    index_name=INDEX_NAME,
    docs=compass_docs,
)
print(f"✅ Inserted {len(compass_docs)} documents successfully")

# Run English Queries Against French Documents

Run English queries against the indexed French documents and display the returned French chunks to demonstrate cross-lingual retrieval.

Cross-lingual retrieval lets you search in one language and surface relevant passages written in another. Here we run **English** queries against the **French** documents indexed in Compass. Compass uses multilingual embeddings so semantically similar content can match even when the query and document languages differ.

Inspect the returned chunks below — note the French `title` fields paired with English query strings.

In [ ]:
queries = [
    "What is the central bank's position on inflation?",
    "What are the rules for interest rate decisions?",
    "How are financial institutions supervised?",
]

inflation_query = queries[0]

for query in queries:
    print(f"Query: {query}\n")
    search_response = compass_client.search_chunks(
        index_name=INDEX_NAME,
        query=query,
    )

    for hit in search_response.hits:
        print(f"  chunk_id: {hit.chunk_id}")
        print(f"  score: {hit.score}")
        print(f"  title: {hit.content['title']}")
        print()
    print("-" * 60 + "\n")

# Fetch hits for the rerank/generate pipeline (always matches inflation_query)
response = compass_client.search_chunks(
    index_name=INDEX_NAME,
    query=inflation_query,
)

# Rerank for Precision

Rerank the retrieved French chunks with a multilingual rerank model to improve precision before answer generation.

Initial retrieval returns a broad set of French chunks ranked by embedding similarity. To sharpen precision before generation, we pass those chunks to Cohere's multilingual **Rerank** model (`rerank-v4.0-pro`). Rerank scores each passage against the English query in a cross-lingual setting, surfacing the most relevant French context for the next step.

In [ ]:
rerank_response = co.rerank(
    model="rerank-v4.0-pro",
    query=inflation_query,
    documents=[r.content["text"] for r in response.hits],
    top_n=3,
)

for result in rerank_response.results:
    hit = response.hits[result.index]
    print(f"index: {result.index}")
    print(f"relevance_score: {result.relevance_score}")
    print(f"title: {hit.content['title']}")
    print()

# Generate Grounded English Answers

Pass the reranked French chunks to Command and generate a grounded English answer with citations back to the source passages.

With the top reranked French passages selected, we call **Command** (`command-a-plus-05-2026`) in grounded RAG mode. The model reads the French source text and produces an **English** answer citing the passages it relied on — demonstrating the full cross-lingual pipeline: English question → French retrieval → English cited response.

In [ ]:
documents = [
    {
        "id": f"doc:{i}",
        "data": {
            "text": response.hits[result.index].content["text"],
            "title": response.hits[result.index].content["title"],
        },
    }
    for i, result in enumerate(rerank_response.results)
]

chat_response = co.chat(
    model="command-a-plus-05-2026",
    messages=[
        {
            "role": "user",
            "content": f"{inflation_query} Answer in English.",
        }
    ],
    documents=documents,
)

for item in chat_response.message.content:
    if hasattr(item, "text") and item.text:
        print(item.text)

if chat_response.message.citations:
    print("\nCITATIONS:")
    for citation in chat_response.message.citations:
        print(citation)

## Understanding the Output

### The Answer
Command generated the English answer exclusively from the French source 
documents — nothing was added from outside the provided context. Each 
bullet point in the answer maps directly to a passage in one of the 
retrieved French chunks.

### The Citations
Each citation object in the output has three parts:

**1. Character positions** (`start` / `end`)
The exact character range in the English answer where this claim appears.

**2. Text** (`text`)
The exact English phrase that was grounded in a source document.

**3. Sources** (`sources`)
The French document that justified the claim, including:
- `id` — which document (`doc:0`, `doc:1`, `doc:2`)
- `title` — the document title in French
- `text` — the full French passage the claim was drawn from

### Why This Matters
In enterprise RAG deployments, citations make every answer auditable. 
A user can trace any claim in the English output back to the exact 
French source passage that justified it — enabling trust, compliance, 
and human review at scale. This is the core value of the 
retrieval → rerank → generate pattern.

# Next Steps

Brief guidance on adapting this notebook to other languages, connecting your own Compass instance, and reusing the pattern in enterprise demos.

### Use your own Compass instance and corpus

Point `COMPASS_INDEX_URL`, `COMPASS_PARSER_URL`, and `COMPASS_BEARER_TOKEN` at your deployment. Replace `sample_documents` with parsed PDFs or HTML via `CompassParserClient`, or build `CompassDocument` objects from your own content. Choose an `INDEX_NAME` that matches your environment.

### Adapt to other language pairs

The same pattern works for any language combination supported by Compass embeddings and `rerank-v4.0-pro` — for example German queries against English policy manuals, or Spanish customer questions against Portuguese knowledge-base articles. Swap the sample corpus and query strings; the retrieval → rerank → generate flow stays the same.

### Reuse in enterprise demos

This three-stage pipeline — semantic search, rerank, grounded generation — maps cleanly to production RAG assistants: Compass handles ingestion and retrieval at scale, Rerank improves hit quality without re-embedding, and Command returns auditable answers with citations. Combine with access controls, per-tenant indexes, and query logging for a complete enterprise search demo.

Note: only `inflation_query` flows through to rerank and generate. To route a different query, update `inflation_query` and re-run the **Run English Queries** cell (to refresh `response`), then re-run rerank and generate. For larger corpora, increase `top_k` in `search_chunks()` for broader recall and `top_n` in `rerank()` for more candidates before generation — `top_k=10` / `top_n=3` are good starting points.